# Statistical Modeling of the Vitamin D Transcriptional Response

This notebook evaluates whether the Vitamin D transcriptional signature
(`core_score`) exhibits a dose–response relationship, while accounting
for variability across cell lines.

A linear mixed-effects model is used to separate the effect of dose
from cell line–specific baseline differences.


In [ ]:
import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf


In [ ]:
# === Input paths (edit to match your repo) ===
META_PATH = "../data/exports/signature_metadata_with_core_score.csv"    # rows = signatures, includes 'sig_id'

assert os.path.exists(META_PATH), f"Missing file: {META_PATH}"

print("META_PATH:", META_PATH)

In [ ]:
meta = pd.read_csv(META_PATH)     # expected: one row per sig_id

print("Metadata shape (rows x cols):", meta.shape)

display(meta.head(3))

In [ ]:
# Required columns
required_cols = ["core_score", "pert_dose", "cell_id", "dose_bin"]
missing = [c for c in required_cols if c not in meta.columns]
assert not missing, f"Missing required columns: {missing}"

df = meta[required_cols].copy()

print(df.shape)
display(df.head())


## Exploratory dose–response patterns

We first visualize the relationship between dose and core_score
to assess global trends and cell line–specific variability.


In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(
    data=df,
    x="pert_dose",
    y="core_score",
    hue="cell_id",
    alpha=0.7
)

sns.regplot(
    data=df,
    x="pert_dose",
    y="core_score",
    scatter=False,
    color="black",
    line_kws={"linewidth": 2}
)

plt.title("Dose–response pattern of the Vitamin D core score")
plt.tight_layout()
plt.show()


The scatter plot shows a clear positive dose–response trend in the Vitamin D
core score across cell lines. While substantial baseline variability is
observed between cellular contexts, the overall relationship between dose
and transcriptional activation remains consistent. This visual pattern
is fully aligned with the mixed-effects model results, supporting a
robust global dose–response effect.


In [ ]:
# Linear mixed-effects model
# Fixed effect: dose
# Random intercept: cell line

model = smf.mixedlm(
    "core_score ~ pert_dose",
    data=df,
    groups=df["cell_id"]
)

result = model.fit()

print(result.summary())


A statistically significant dose–response relationship is observed between Vitamin D dose and the transcriptional core score, after accounting for cell line–specific baseline variability.